<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# `cute.Tensor` ↔ `cutlass.Array` interop (with `make_array_view`)

The DSL gives you two handles onto the same GPU memory. A `cute.Tensor` is a pointer composed with a
CuTe **layout** — it is what `cute.runtime.from_dlpack` produces, and the type that the layout
algebra (`local_tile`, `local_partition`, ...) and copy atoms operate on. A `cutlass.Array` is a
pointer plus a **flat** shape / strides / dtype / address space — the type from the
`01_array_concepts` notebook, with its `a[idx:V]` "slice = vectorized load/store" idiom and the
`cutlass.Array(dtype, shape, space=...)` allocator.

**Both** support element access (`t[(i, j)]`, `a[i, j]`) and both have a notion of slicing — but
the slices mean different things (a sub-tensor *view* vs. a `V`-wide *vector value*). Section 1
lays the two side by side.

So far you have only seen the `cutlass.Array` side, because annotating a parameter `cutlass.Array`
and passing a `from_dlpack` tensor makes the host entry convert for you. This notebook pulls that
conversion into the open: `cutlass.make_array_view` takes a `cute.Tensor` and hands you a
`cutlass.Array` over the **same memory, no copy** — so a tensor you already hold (a kernel parameter,
a tile you sliced out) can use the Array idioms whenever they fit better.

**You'll learn:** what a `cute.Tensor` (pointer ∘ CuTe layout — `from_dlpack`, layout algebra,
copy atoms) and a `cutlass.Array` (pointer + flat shape/strides — `a[i, j]`, `a[idx:V]` vector
slices, `space=` allocation) each support, where they overlap, where they differ, and how
`cutlass.make_array_view(tensor)` bridges the former to the latter with zero copy.

**Runs on:** any CUDA GPU. **Prereq:** the `01_array_concepts` notebook (`cutlass.Array` indexing).

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

## 1. Two handles onto GPU memory

Neither type is a superset of the other. Both address memory element by element; they differ in
what their *layout* can express, what a *slice* returns, and which APIs consume them.

| | `cute.Tensor` | `cutlass.Array` |
|---|---|---|
| what it is | pointer (iterator) ∘ CuTe **layout**: `T(c) = *(ptr + L(c))` | pointer + **flat** shape, strides, dtype, address space |
| layout can be | any CuTe layout: nested / hierarchical modes, composed (swizzled), static or dynamic | flat (non-nested) shape/strides only — `make_array_view` rejects nested layouts |
| produced by | `cute.runtime.from_dlpack`, `cute.make_tensor`, slicing or tiling another tensor | `cutlass.Array(dtype, shape, space=...)`, `cutlass.make_array_view(t)`, host-entry conversion of a tensor passed to a `cutlass.Array` parameter |
| element load / store | `t[i]` (linear index, mapped through the layout), `t[(i, j)]` (coordinate) | `a[i]` (flat index), `a[i, j]` (stride-aware) |
| what a slice is | `t[(i, None)]` / `cute.slice_` keeps whole modes and returns a **sub-tensor view** — no data moves | `a[idx:V]` returns a **`cutlass.Vector` of `V` contiguous elements** — a vectorized load; `V` is a *count*, not a stop index |
| vector load / store | `t.load()` / `t.store(...)` on the whole (static-layout) tensor as a `TensorSSA` | `a[idx:V]` / `a[idx:V] = vec`, with an optional `cutlass.align(16)` hint for wide transactions |
| layout algebra | `cute.local_tile`, `cute.local_partition`, `cute.zipped_divide`, `cute.composition`, `cute.recast_tensor` | none — only `a.subview(n)` (offset Array) and `a.data_ptr(n)` (raw pointer) |
| copy engines | `cute.copy` with copy atoms, incl. TMA (`make_tiled_tma_atom`) | the `05_tma_load` route: `cuda.create_tensor_map_tiled_from_view(a)` and `prims.cp_async_bulk_tensor_*` accept an Array directly |
| allocation | `cute.make_rmem_tensor` (registers), `SmemAllocator` (shared) | `cutlass.Array(..., space=rmem / smem / gmem / cmem)` |

Rule of thumb: stay on `cute.Tensor` while you are *reshaping* memory (tiling, partitioning,
swizzling, feeding copy atoms); use `cutlass.Array` when you want the flat, CUDA-C-like access
of the `01_array_concepts` notebook — especially the `a[idx:V]` window idiom, whose result is a
`cutlass.Vector` you can do register arithmetic on.

`cutlass.make_array_view(t)` reinterprets a tensor's base pointer and (flat) layout as a
`cutlass.Array` aliasing the **same** memory — no allocation, no copy, just a different handle onto
the same bytes.

## 2. The bridge, inside the kernel

This kernel takes its parameters as `cute.Tensor` on purpose — the layout-carrying type
`cute.runtime.from_dlpack` produces. It *could* read elements straight off the tensor with
`inp[idx]`; instead it wants the `01_array_concepts` window idiom, so it first builds a view with
`make_array_view`. From then on it is Array indexing: the slice `a[idx:V]` is a vectorized
load/store of `V` contiguous elements (a `cutlass.Vector`), which is what lets `* 2.0` operate on
the whole window at once.

To make the aliasing concrete, thread 0 reads one element through the view both before and after
the store. The "after" value is `2 *` the "before" value — proof the view writes straight into the
tensor's memory.

In [ ]:
@cute.kernel
def scale_kernel(
    inp: cute.Tensor,
    out: cute.Tensor,
    N: cutlass.Int32,
    V: cutlass.Constexpr,
):
    # Step 1. This thread owns the V-wide element window starting at idx.
    tx, _, _ = cute.arch.thread_idx()
    bx, _, _ = cute.arch.block_idx()
    bdx, _, _ = cute.arch.block_dim()
    idx = (bx * bdx + tx) * V

    # Step 2. The bridge: alias each cute.Tensor as a cutlass.Array over the same
    # memory (base pointer + flat layout, no copy). We could read inp[idx] off the
    # tensor directly; the Array view is what gives us the a[idx:V] vector slice.
    inp_arr = cutlass.make_array_view(inp)
    out_arr = cutlass.make_array_view(out)

    if idx < N:
        # Step 3. Scale the window. The slice is a vectorized load/store; the second
        # number is a COUNT of V elements. Thread 0 reads one element through the view
        # before and after to prove the view aliases the tensor's memory (out is 2x in).
        if idx == 0:
            cute.printf(f"[thread 0] in : inp_arr[0]={inp_arr[0]}")

        out_arr[idx:V] = inp_arr[idx:V] * 2.0

        if idx == 0:
            cute.printf(f"[thread 0] out: out_arr[0]={out_arr[0]}  (= 2 * in)")

## 3. Launch: one thread per V elements

Identical to the `01_array_concepts` launch. We need `N / V` threads (one per window), in a
256-wide 1-D block with the grid rounded up.

In [ ]:
@cute.jit
def scale(
    inp: cute.Tensor,
    out: cute.Tensor,
    N: cutlass.Int32,
    V: cutlass.Constexpr,
):
    block = (256, 1, 1)
    threads = N // V  # one thread per V-wide window
    grid = ((threads + block[0] - 1) // block[0], 1, 1)
    scale_kernel(inp, out, N, V).launch(grid=grid, block=block)

## 4. Run it and check against PyTorch

Because the parameters are annotated `cute.Tensor`, we pass the layout-carrying values
`cute.runtime.from_dlpack` returns directly — no host-entry conversion — and the kernel does the
`make_array_view` bridge itself. Then we check against `2 * inp` on the host.

In [ ]:
# from_dlpack yields cute.Tensors (full layout preserved); the kernel
# make_array_view's them into cutlass.Arrays for the a[idx:V] vector-slice idiom.
N, V = 1 << 20, 4
inp = torch.randn(N, dtype=torch.float32, device="cuda")
out = torch.zeros(N, dtype=torch.float32, device="cuda")

scale(cute.runtime.from_dlpack(inp), cute.runtime.from_dlpack(out), N, V)

# Verify against PyTorch (compare on host).
torch.testing.assert_close(out.cpu(), (inp * 2.0).cpu(), atol=1e-5, rtol=1e-5)
print("PASS")

# Expected output:
# PASS

## Try it yourself

1. Re-annotate the parameters `cutlass.Array` and pass the same `from_dlpack` tensors. It still
   works — the host entry runs this same bridge for you. So when is an *explicit* `make_array_view`
   necessary? (Hint: when the `cute.Tensor` only comes into existence *inside* the kernel — e.g. a
   tile from `cute.local_tile` or a `cute.slice_` — and you want Array-style access to it.)
2. Skip the bridge: replace `inp_arr[idx:V]` with a loop over `inp[idx + k]` / `out[idx + k]`
   for `k in range(V)` — element access works on the tensor directly. Then compare the PTX
   (`cute.compile[cute.KeepPTX]`, as in `01_array_concepts` §4): does the per-element loop
   still produce the one wide load/store per window that the `a[idx:V]` slice does?
3. `make_array_view` is zero-copy. Confirm that writing through `out_arr` really changes `out` —
   the view and the tensor share storage.
4. Print `inp_arr.shape` / `inp_arr.strides` / `inp_arr.dtype`: the view exposes the same
   compile-time layout facts as the `cutlass.Array`s in the `01_array_concepts` notebook.